# 第一章：从图像到序列 — Vision Transformer

> **前置要求**：本书假设你已读完《Build a Large Language Model from Scratch》，
> 熟悉 Transformer、多头注意力、残差连接等概念。
> 这些原理本章不重复讲解，只聚焦于图像模态带来的新问题。

---

## 本章目标

从零实现 **Vision Transformer (ViT)**，它是几乎所有现代多模态模型的图像编码器
（CLIP、LLaVA、Flamingo、GPT-4V 都用了类似的结构）。

读完本章，你将理解：
1. 为什么不能直接把像素送进 Transformer
2. **Patch Embedding**：把图像变成 token 序列的方法
3. ViT 与 GPT 的核心区别：**双向注意力 vs 因果注意力**
4. `[CLS]` token 如何聚合整张图像的信息
5. 把所有组件组装成完整 ViT，并在合成数据上训练验证

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), '.'))
os.makedirs('figures', exist_ok=True)

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

---
## 1.1 为什么图像需要特殊处理？

你可能会想：Transformer 能处理任意序列，直接把图像的每个像素当成一个 token 不就行了？

我们来算一下代价：

In [ ]:
H, W, C = 224, 224, 3

# 方案一：每个像素是一个 token
n_pixel_tokens = H * W   # 忽略通道，已经很多了
attn_ops = n_pixel_tokens ** 2
print("=== 方案一：像素级 token ===")
print(f"  Token 数量    : {n_pixel_tokens:,}")
print(f"  注意力矩阵    : {n_pixel_tokens} × {n_pixel_tokens} = {attn_ops:,} 个值")
print(f"  约等于        : {attn_ops/1e9:.1f} 十亿 — 每张图都会 OOM\n")

# 方案二：16×16 patch
patch_size = 16
n_patches = (H // patch_size) * (W // patch_size)
attn_ops_vit = n_patches ** 2
print("=== 方案二：Patch token (ViT) ===")
print(f"  Patch 数量    : ({H}/{patch_size}) × ({W}/{patch_size}) = {n_patches}")
print(f"  注意力矩阵    : {n_patches} × {n_patches} = {attn_ops_vit:,} 个值")
print(f"  压缩比        : {attn_ops // attn_ops_vit}× 更小 ✓")
print(f"\n每个 patch 的原始大小: {patch_size}×{patch_size}×{C} = {patch_size*patch_size*C} 个值")

ViT 的核心 insight 只有一句话：

> **把图像切成 16×16 的小块（patch），每块当成一个 token，就能用标准 Transformer 处理了。**

这个想法在 2020 年的论文 *"An Image is Worth 16x16 Words"* 中提出，标题本身就是这个 idea。

下面我们来看这个过程：

In [ ]:
# 展示图像切块流程
import sys; sys.path.insert(0, '..')
from multimodal_from_scratch.figures import draw_patch_splitting
fig = draw_patch_splitting(save_path='figures/ch01_patch_split.png')
plt.show()

---
## 1.2 Patch Embedding：从零实现

### 1.2.1 朴素实现：手动切块

先用最直白的方式实现，理解原理：

In [ ]:
def extract_patches_naive(img: torch.Tensor, patch_size: int) -> torch.Tensor:
    """
    最朴素的 patch 提取：用循环逐个切块。
    img: (B, C, H, W)
    返回: (B, n_patches, C * patch_size * patch_size)
    """
    B, C, H, W = img.shape
    patches = []
    for row in range(0, H, patch_size):
        for col in range(0, W, patch_size):
            # 取出一个 patch: (B, C, patch_size, patch_size)
            patch = img[:, :, row:row+patch_size, col:col+patch_size]
            # 展平为向量: (B, C * patch_size * patch_size)
            patch = patch.reshape(B, -1)
            patches.append(patch)
    # 堆叠成序列: (B, n_patches, C*P*P)
    return torch.stack(patches, dim=1)

# 验证
img = torch.randn(2, 3, 32, 32)   # batch=2, 32×32 图像
patches = extract_patches_naive(img, patch_size=8)
n_patches = (32 // 8) ** 2
patch_dim  = 3 * 8 * 8
print(f"输入形状:  {img.shape}       (B, C, H, W)")
print(f"输出形状:  {patches.shape}  (B, n_patches, C*P*P)")
print(f"         = (2, {n_patches}, {patch_dim})")

### 1.2.2 优雅实现：用 Conv2d 代替循环

朴素实现有个问题：Python 循环很慢，而且切块之后还需要一个线性层做投影。

有个漂亮的技巧：**`kernel_size=patch_size, stride=patch_size` 的 Conv2d 恰好等价于「切块 + 线性投影」**。

原理：卷积核在不重叠的 patch 上滑动（stride = kernel_size），每个 patch 输出一个 `embed_dim` 维向量——这正是线性投影在做的事情。

In [ ]:
# 验证：Conv2d 与 朴素实现 + Linear 等价
B, C, H, W = 1, 3, 32, 32
P = 8   # patch size
D = 64  # embed dim

img = torch.randn(B, C, H, W)

# 方法 A：朴素切块 + Linear
linear = nn.Linear(C * P * P, D, bias=False)
patches_naive = extract_patches_naive(img, P)          # (B, n, C*P*P)
out_A = linear(patches_naive)                           # (B, n, D)

# 方法 B：Conv2d（把 linear 的权重复制过来以验证等价性）
conv = nn.Conv2d(C, D, kernel_size=P, stride=P, bias=False)
# Conv2d weight shape: (out_channels, in_channels, kH, kW) = (D, C, P, P)
# 重新排列使其等于 linear 的权重
with torch.no_grad():
    conv.weight.copy_(linear.weight.reshape(D, C, P, P))

out_B = conv(img)              # (B, D, H/P, W/P)
out_B = out_B.flatten(2)       # (B, D, n_patches)
out_B = out_B.transpose(1, 2)  # (B, n_patches, D)

print(f"方法 A (朴素+Linear) 输出形状: {out_A.shape}")
print(f"方法 B (Conv2d)      输出形状: {out_B.shape}")
print(f"结果是否一致: {torch.allclose(out_A, out_B, atol=1e-5)}  ← 完全等价！")

### 1.2.3 实现 PatchEmbedding 类

有了以上理解，我们来写干净的实现：

In [ ]:
class PatchEmbedding(nn.Module):
    """
    将图像切分成不重叠的 patch，并线性投影到 embed_dim 维空间。
    一个 Conv2d 同时完成「切块」和「线性投影」两步。
    """

    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        assert img_size % patch_size == 0, \
            f"图像尺寸 {img_size} 必须能被 patch 尺寸 {patch_size} 整除"

        self.patch_size = patch_size
        self.n_patches  = (img_size // patch_size) ** 2

        # kernel_size=stride=patch_size → 不重叠地滑过每个 patch
        self.proj = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )

    def forward(self, x):
        # x: (B, C, H, W)
        x = self.proj(x)       # → (B, embed_dim, H/P, W/P)
        x = x.flatten(2)       # → (B, embed_dim, n_patches)
        x = x.transpose(1, 2)  # → (B, n_patches, embed_dim)
        return x


# 验证 shape 变换
patch_embed = PatchEmbedding(img_size=224, patch_size=16, in_channels=3, embed_dim=768)
batch = torch.randn(2, 3, 224, 224)
out   = patch_embed(batch)

print(f"输入: {batch.shape}")
print(f"  ↓ Conv2d(kernel=16, stride=16)")
print(f"  ↓ flatten + transpose")
print(f"输出: {out.shape}")
print(f"      (batch=2, n_patches={(224//16)**2}, embed_dim=768)")
print(f"\n参数量: {sum(p.numel() for p in patch_embed.parameters()):,}")
print(f"  = embed_dim × (C×P×P + 1) = 768 × (3×16×16 + 1) = 768 × 769")

---
## 1.3 位置编码

### 为什么图像也需要位置信息？

Transformer 的注意力机制本身是**位置无关的**——同样的一组 patch，无论打乱顺序还是按原顺序输入，注意力的计算方式完全相同。

但位置对图像至关重要：左上角的 patch 和右下角的 patch 含义不同。
下面演示打乱顺序的效果：

In [ ]:
# 创建一张有结构的合成图像（左半蓝、右半橙）
img_demo = torch.zeros(1, 3, 32, 32)
img_demo[0, 2, :, :16] = 1.0   # 左半蓝色（B通道）
img_demo[0, 0, :, 16:] = 1.0   # 右半红色（R通道）
img_demo[0, 1, :, 16:] = 0.5   # 右半加绿 → 橙色

pe = PatchEmbedding(img_size=32, patch_size=8, in_channels=3, embed_dim=32)
tokens = pe(img_demo)   # (1, 16, 32)

# 打乱 patch 顺序
shuffle_idx = torch.randperm(tokens.shape[1])
tokens_shuffled = tokens[:, shuffle_idx, :]

fig, axes = plt.subplots(1, 3, figsize=(12, 3))

axes[0].imshow(img_demo[0].permute(1,2,0).clamp(0,1))
axes[0].set_title('原始图像\n(左蓝右橙)', fontsize=11)
axes[0].axis('off')

axes[1].imshow(tokens[0].detach(), aspect='auto', cmap='RdBu')
axes[1].set_title(f'正确顺序 token 序列\n形状 {list(tokens.shape[1:])}', fontsize=11)
axes[1].set_xlabel('embed_dim=32'); axes[1].set_ylabel('patch 位置')

axes[2].imshow(tokens_shuffled[0].detach(), aspect='auto', cmap='RdBu')
axes[2].set_title(f'打乱顺序后\n模型无法区分！', fontsize=11)
axes[2].set_xlabel('embed_dim=32')

plt.suptitle('没有位置编码，Transformer 看到的是同一组向量', fontsize=12)
plt.tight_layout()
plt.show()

ViT 使用**可学习的位置嵌入**（而非 GPT 中的固定正弦位置编码）。
实验表明，在固定分辨率的图像 token 上，可学习方式效果相当甚至更好。

In [ ]:
class PositionalEmbedding(nn.Module):
    """
    可学习的 1D 位置嵌入。
    形状: (1, n_patches + 1, embed_dim)  (+1 是为 [CLS] token 留位置)
    """

    def __init__(self, n_patches, embed_dim):
        super().__init__()
        # 初始化为接近 0 的小值，训练过程中学习位置信息
        self.pos_embed = nn.Parameter(
            torch.zeros(1, n_patches + 1, embed_dim)  # +1 for [CLS]
        )
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        # x: (B, n_patches+1, embed_dim)
        # 直接相加，广播到 batch 维度
        return x + self.pos_embed


n_patches = (224 // 16) ** 2   # = 196
pos_embed = PositionalEmbedding(n_patches=n_patches, embed_dim=768)

# 模拟 [CLS] + patches 的序列
x = torch.randn(2, n_patches + 1, 768)  # +1 for CLS
out = pos_embed(x)

print(f"位置嵌入参数形状: {pos_embed.pos_embed.shape}")
print(f"输入序列形状:  {x.shape}")
print(f"加完位置后:   {out.shape}  (形状不变，每个位置有了独特的偏移)")

---
## 1.4 [CLS] Token：全局图像表示

在 ViT 中，我们在 patch 序列前面加一个特殊的 `[CLS]` token（Classification token，来自 BERT）。

**为什么需要它？**

Transformer 输出的是一个序列，每个 token 都有对应输出。但我们通常需要一个**单一向量**来代表整张图像。有两个选择：

| 方法 | 说明 | 问题 |
|------|------|------|
| 对所有 patch 输出取平均 | 简单 | 丢失了不同 patch 的权重差异 |
| 额外加一个 [CLS] token | [CLS] 可自由地与所有 patch 做注意力，聚合全局信息 | 需要多一个 token |

ViT 选择了 `[CLS]` 方案，实验证明它比平均池化效果更好。

`[CLS]` 是一个**可学习的向量**，初始为随机值，通过反向传播学习成为「好的全局聚合器」。

In [ ]:
# 演示 [CLS] token 的拼接过程
B, n_patches, embed_dim = 2, 196, 768

# 1. patch tokens，来自 PatchEmbedding
patch_tokens = torch.randn(B, n_patches, embed_dim)
print(f"Step 1 — Patch tokens:   {patch_tokens.shape}")

# 2. 可学习的 [CLS] token（每个样本共享同一个初始向量）
cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
# expand 到 batch 维度，注意：这里不使用 repeat（不拷贝数据）
cls_tokens = cls_token.expand(B, -1, -1)  # (B, 1, embed_dim)
print(f"Step 2 — CLS token:      {cls_tokens.shape}")

# 3. 拼接：[CLS] 在最前面
full_seq = torch.cat([cls_tokens, patch_tokens], dim=1)  # (B, 197, embed_dim)
print(f"Step 3 — Full sequence:  {full_seq.shape}")
print(f"         = (B, 1+{n_patches}, {embed_dim})")
print(f"\n经过 Transformer 后，取 full_seq[:, 0, :] 作为图像表示")
print(f"[CLS] 输出形状: {full_seq[:, 0, :].shape}  ← 每张图一个向量")

---
## 1.5 ViT 的注意力：与 GPT 的唯一区别

如果你读完了原书，注意力机制的数学原理你已经完全掌握了。

ViT 和 GPT 的注意力模块**几乎完全相同**，唯一区别是：

| | GPT（语言模型） | ViT（图像编码器） |
|--|--|--|
| 注意力类型 | **因果**（causal） | **双向**（bidirectional） |
| 是否有掩码 | ✓ 上三角掩码，阻止看未来 | ✗ 无掩码，所有 patch 互看 |
| 原因 | 语言生成必须从左到右 | 图像没有时间顺序，patch 互相参考 |

这一点非常关键，下面的图对比了两种注意力模式：

In [ ]:
from multimodal_from_scratch.figures import draw_attention_masks
fig = draw_attention_masks(save_path='figures/ch01_attn_masks.png')
plt.show()

去掉因果掩码，代码上只需改一行：

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """
    双向多头自注意力（ViT 版本）。
    与原书的 CausalMultiHeadAttention 相比，唯一区别是：
    没有 register_buffer('causal_mask', ...) 以及 masked_fill 那一行。
    """

    def __init__(self, embed_dim, num_heads, dropout=0.0):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.scale     = self.head_dim ** -0.5  # 1/√d_k，防止点积过大

        self.qkv      = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.drop     = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape

        # Q, K, V 一次计算，再拆分到各个头
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        # q, k, v: (B, num_heads, N, head_dim)

        # 缩放点积注意力
        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, heads, N, N)
        # ← 这里没有 masked_fill！所有位置都可以相互注意
        attn = F.softmax(attn, dim=-1)
        attn = self.drop(attn)

        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out)


# 验证
attn = MultiHeadSelfAttention(embed_dim=192, num_heads=3)
x = torch.randn(2, 197, 192)   # (B, n_patches+1, D)
out = attn(x)
print(f"注意力输入: {x.shape}")
print(f"注意力输出: {out.shape}  （形状不变，每个 token 更新了表示）")

---
## 1.6 组装完整 ViT

现在把所有组件拼起来。ViT 的 Transformer Block 结构与 GPT 完全相同：

```
x = x + Attention(LayerNorm(x))   # 残差 + 注意力
x = x + FFN(LayerNorm(x))         # 残差 + 前馈网络
```

唯一区别是注意力没有因果掩码。我们复用之前实现的 `MultiHeadSelfAttention`：

In [ ]:
class ViTBlock(nn.Module):
    """一个 Transformer Encoder Block（Pre-LayerNorm 版本）。"""

    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        # FFN：两层线性，中间 GELU，隐层维度 = embed_dim × mlp_ratio
        hidden = int(embed_dim * mlp_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))  # 注意力残差
        x = x + self.ffn(self.norm2(x))   # FFN 残差
        return x

In [ ]:
class VisionTransformer(nn.Module):
    """
    完整 Vision Transformer。

    forward() 返回一个 dict：
      'cls'    : (B, D)      — [CLS] token，全局图像表示
      'tokens' : (B, N, D)   — patch tokens，含空间信息
      'all'    : (B, N+1, D) — cls + patches

    在 CLIP 预训练（第2章）中使用 'cls'；
    在 VLM（第3章）中使用 'tokens' 保留空间细节。
    """

    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 embed_dim=768, depth=12, num_heads=12,
                 mlp_ratio=4.0, dropout=0.0):
        super().__init__()

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n_patches = self.patch_embed.n_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = PositionalEmbedding(n_patches, embed_dim)
        self.drop      = nn.Dropout(dropout)

        self.blocks = nn.Sequential(
            *[ViTBlock(embed_dim, num_heads, mlp_ratio, dropout)
              for _ in range(depth)]
        )
        self.norm = nn.LayerNorm(embed_dim)

        # 权重初始化（与原论文一致）
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        B = x.shape[0]

        # 1. 图像 → patch tokens
        x = self.patch_embed(x)                          # (B, N, D)

        # 2. 拼接 [CLS]
        cls = self.cls_token.expand(B, -1, -1)           # (B, 1, D)
        x   = torch.cat([cls, x], dim=1)                 # (B, N+1, D)

        # 3. 加位置编码
        x = self.pos_embed(x)
        x = self.drop(x)

        # 4. Transformer blocks
        x = self.blocks(x)
        x = self.norm(x)

        return {'cls': x[:, 0], 'tokens': x[:, 1:], 'all': x}

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
# 对比不同规格的 ViT
configs = {
    'ViT-Tiny' : dict(embed_dim=192, depth=12, num_heads=3),
    'ViT-Small': dict(embed_dim=384, depth=12, num_heads=6),
    'ViT-Base' : dict(embed_dim=768, depth=12, num_heads=12),
    'ViT-Large': dict(embed_dim=1024, depth=24, num_heads=16),
}

print(f"{'模型':<12} {'embed_dim':>10} {'depth':>6} {'heads':>6} {'参数量':>12}")
print('-' * 52)
for name, cfg in configs.items():
    m = VisionTransformer(**cfg)
    print(f"{name:<12} {cfg['embed_dim']:>10} {cfg['depth']:>6} "
          f"{cfg['num_heads']:>6} {m.count_parameters():>12,}")

In [ ]:
# 完整前向传播验证
vit = VisionTransformer(img_size=224, patch_size=16, embed_dim=192,
                        depth=12, num_heads=3)  # ViT-Tiny

images = torch.randn(2, 3, 224, 224)
out    = vit(images)

print("=== ViT 前向传播各阶段形状 ===")
print(f"输入图像         : {images.shape}")
print(f"  → patch tokens : (B, {(224//16)**2}, 192)  经过 PatchEmbedding")
print(f"  → + CLS token  : (B, {(224//16)**2+1}, 192)")
print(f"  → + 位置编码   : 形状不变")
print(f"  → 12 ViTBlocks : 形状不变")
print(f"\n最终输出:")
print(f"  cls    (全局)  : {out['cls'].shape}   ← 送给分类头 或 CLIP 投影")
print(f"  tokens (局部)  : {out['tokens'].shape}  ← 送给 VLM 解码器")
print(f"  all            : {out['all'].shape}")

---
## 1.7 让模型学点东西：图像分类演示

现在我们训练一个玩具 ViT 分类器，验证整个流程真的能 work。

### 合成数据集：4 类几何图案

我们构造 4 种视觉可区分的图案，每类各 200 张：

In [ ]:
def make_synthetic_dataset(n_per_class=200, img_size=32, seed=0):
    """
    生成 4 类合成图像：
      0 — 纯红色
      1 — 纯蓝色
      2 — 水平条纹（上红下蓝）
      3 — 垂直条纹（左红右蓝）
    """
    torch.manual_seed(seed)
    imgs, labels = [], []
    H = W = img_size

    for label in range(4):
        for _ in range(n_per_class):
            img = torch.zeros(3, H, W)
            noise = torch.randn(3, H, W) * 0.08   # 少量噪声

            if label == 0:   # 纯红
                img[0] = 1.0
            elif label == 1: # 纯蓝
                img[2] = 1.0
            elif label == 2: # 水平条纹
                img[0, :H//2, :] = 1.0   # 上半红
                img[2, H//2:, :] = 1.0   # 下半蓝
            else:            # 垂直条纹
                img[0, :, :W//2] = 1.0   # 左半红
                img[2, :, W//2:] = 1.0   # 右半蓝

            imgs.append((img + noise).clamp(0, 1))
            labels.append(label)

    return torch.stack(imgs), torch.tensor(labels)


X, y = make_synthetic_dataset(n_per_class=200, img_size=32)
print(f"数据集: {X.shape}, 标签: {y.shape}")

# 可视化每类样本
class_names = ['纯红', '纯蓝', '水平条纹', '垂直条纹']
fig, axes = plt.subplots(1, 4, figsize=(10, 2.5))
for i in range(4):
    idx = (y == i).nonzero()[0].item()
    axes[i].imshow(X[idx].permute(1,2,0))
    axes[i].set_title(f'类别 {i}\n{class_names[i]}', fontsize=10)
    axes[i].axis('off')
plt.suptitle('合成训练数据（每类 200 张）', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, random_split

# 划分训练/测试集
dataset = TensorDataset(X, y)
n_train = int(0.8 * len(dataset))
train_set, test_set = random_split(dataset, [n_train, len(dataset) - n_train],
                                   generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=64)

print(f"训练集: {len(train_set)} 张，测试集: {len(test_set)} 张")

# 建立带分类头的小 ViT
class ViTClassifier(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        # img_size=32, patch_size=8 → 16 个 patch
        self.vit  = VisionTransformer(img_size=32, patch_size=8,
                                      embed_dim=64, depth=3, num_heads=4)
        self.head = nn.Linear(64, num_classes)

    def forward(self, x):
        cls_out = self.vit(x)['cls']   # (B, 64)
        return self.head(cls_out)      # (B, 4)

model = ViTClassifier()
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 训练循环
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

train_losses, test_accs = [], []

for epoch in range(30):
    # --- 训练 ---
    model.train()
    epoch_loss = 0
    for xb, yb in train_loader:
        logits = model(xb)
        loss   = F.cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    # --- 评估 ---
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total   += len(yb)
    acc = correct / total
    test_accs.append(acc)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/30 | loss={avg_loss:.4f} | test acc={acc:.1%}")

In [ ]:
# 绘制训练曲线
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(range(1, 31), train_losses, 'b-o', markersize=4, label='训练 Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('训练损失下降曲线', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(range(1, 31), [a*100 for a in test_accs], 'g-o', markersize=4, label='测试准确率')
ax2.axhline(25, color='red', linestyle='--', alpha=0.5, label='随机猜测 (25%)')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_ylim(0, 105)
ax2.set_title('测试集准确率', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.suptitle('ViT 在合成分类任务上的训练结果', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/ch01_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\n最终测试准确率: {test_accs[-1]:.1%}")
print(f"（随机猜测基线: 25%）")

---
## 本章小结

### 我们从零构建了什么

| 组件 | 作用 | 关键细节 |
|------|------|----------|
| `PatchEmbedding` | 图像 → token 序列 | Conv2d 代替循环，一步完成切块+投影 |
| `PositionalEmbedding` | 注入位置信息 | 可学习参数，初始化接近 0 |
| `[CLS] token` | 全局图像表示 | prepend 到序列头部，取其输出 |
| `MultiHeadSelfAttention` | 图像内部关系建模 | **无因果掩码**，与 GPT 最大的不同 |
| `ViTBlock` | 注意力 + FFN | Pre-LayerNorm，与原书 GPT Block 相同 |
| `VisionTransformer` | 完整编码器 | 输出 `cls`（全局）和 `tokens`（局部）|

### 与原书 GPT 的对比

```
GPT：  [t1, t2, t3, t4] → 因果注意力 → 每个 token 预测下一个
ViT：  [CLS, p1, p2, ... p196] → 双向注意力 → CLS 聚合整张图像
```

### 下一章预告

现在 ViT 只是一个图像分类器。在**第 2 章**，我们要用 **CLIP** 对比学习
让 ViT 不再只看「这是猫还是狗」，而是理解「这张图和这句话匹配吗？」——
这是通往多模态的关键一步。